In [4]:
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset

path = hf_hub_download(
    repo_id="Vishal0407/GSM8K_TAMIL",
    filename="gsm8k_tamil.json",
    repo_type="dataset",
)

with open(path, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("JSON type:", type(raw).__name__)
if isinstance(raw, list):
    print("First record:", raw[0])
elif isinstance(raw, dict):
    print("Top-level keys:", list(raw.keys()))

JSON type: list
First record: {'question': 'ஏப்ரல் மாதத்தில் நடாலியா தனது 48 நண்பர்களுக்கு கிளிப்களை விற்றார், பின்னர் அவர் மே மாதத்தில் பாதி கிளிப்களை விற்றார். ஏப்ரல் மற்றும் மே மாதங்களில் நடாலியா முற்றிலும் எத்தனை கிளிப்களை விற்றது?', 'answer': 'நடாலியா 48/2 = << 48/2 = 24 >> மே மாதத்தில் 24 கிளிப்புகள் விற்றது.\nநடாலியா 48+24 = << 48+24 = 72 >> ஏப்ரல் மற்றும் மே மாதங்களில் 72 கிளிப்புகள் விற்றது.\n#### 72'}


In [5]:
from datasets import Dataset

ds = Dataset.from_list(raw)

print("Rows:", len(ds))
print("Columns:", ds.column_names)
print("\nQuestion:", ds[0]["question"])
print("\nGold answer:", ds[0]["answer"])

Rows: 8792
Columns: ['question', 'answer']

Question: ஏப்ரல் மாதத்தில் நடாலியா தனது 48 நண்பர்களுக்கு கிளிப்களை விற்றார், பின்னர் அவர் மே மாதத்தில் பாதி கிளிப்களை விற்றார். ஏப்ரல் மற்றும் மே மாதங்களில் நடாலியா முற்றிலும் எத்தனை கிளிப்களை விற்றது?

Gold answer: நடாலியா 48/2 = << 48/2 = 24 >> மே மாதத்தில் 24 கிளிப்புகள் விற்றது.
நடாலியா 48+24 = << 48+24 = 72 >> ஏப்ரல் மற்றும் மே மாதங்களில் 72 கிளிப்புகள் விற்றது.
#### 72


In [12]:
import re
from decimal import Decimal, InvalidOperation

def normalize_number(value):
    if value is None:
        return None

    value = str(value).strip().replace(",", "")

    try:
        number = Decimal(value)
        if not number.is_finite():
            return None
        return format(number.normalize(), "f")
    except InvalidOperation:
        return None


def extract_gold_answer(text):
    matches = re.findall(
        r"####\s*(-?\d[\d,]*(?:\.\d+)?)",
        str(text)
    )
    return normalize_number(matches[-1]) if matches else None


def extract_predicted_answer(text):
    text = str(text)

    patterns = [
    r"####\s*<?\s*(-?\d[\d,]*(?:\.\d+)?)\s*>?",
    r"\\boxed\{\s*(-?\d[\d,]*(?:\.\d+)?)\s*\}",
    r"(?:final answer|the answer is|answer)\s*[:：]?\s*"
    r"(-?\d[\d,]*(?:\.\d+)?)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        if matches:
            return normalize_number(matches[-1])

    matches = re.findall(r"-?\d[\d,]*(?:\.\d+)?", text)
    return normalize_number(matches[-1]) if matches else None


print("Gold answer:", extract_gold_answer(ds[0]["answer"]))
assert extract_gold_answer(ds[0]["answer"]) == "72"

Gold answer: 72


In [9]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "Qwen/Qwen3.5-2B"

assert torch.cuda.is_available(), "Enable a GPU runtime in Colab."

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

print("GPU:", torch.cuda.get_device_name(0))
print("Model loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

GPU: Tesla T4
Model loaded successfully.


In [13]:
question = ds[0]["question"]

prompt = f"""கீழே உள்ள கணிதக் கேள்வியைத் தமிழில் படிப்படியாகத் தீர்க்கவும்.
இறுதி எண்ணியல் விடையை #### <number> என்ற வடிவில் கொடுக்கவும்.

கேள்வி:
{question}
"""

messages = [{"role": "user", "content": prompt}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=768,
        do_sample=False,
        use_cache=True,
    )

response = processor.decode(
    output_ids[0, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("Model output:\n", response)
print("\nPredicted answer:", extract_predicted_answer(response))
print("Gold answer:", extract_gold_answer(ds[0]["answer"]))

Model output:
 கீழே உள்ள கேள்வியைத் தமிழில் படிப்படியாகத் தீர்க்கவும்:

**கேள்வி:**
ஏப்ரல் மாதத்தில் நடாலியா தனது 48 நண்பர்களுக்கு கிளிப்களை விற்றார், பின்னர் அவர் மே மாதத்தில் பாதி கிளிப்களை விற்றார். ஏப்ரல் மற்றும் மே மாதங்களில் நடாலியா முற்றிலும் எத்தனை கிளிப்களை விற்றது?

---

### **படி 1: ஏப்ரல் மாதத்தில் விற்ற கிளிப்களின் எண்ணிக்கை**
கேள்வியில் கொடுக்கப்பட்டுள்ள தகவலப்படி, ஏப்ரல் மாதத்தில் நடாலியா 48 நண்பர்களுக்கு கிளிப்களை விற்றார்.
எனவே, ஏப்ரல் மாதத்தில் விற்ற கிளிப்களின் எண்ணிக்கை = **48**.

### **படி 2: மே மாதத்தில் விற்ற கிளிப்களின் எண்ணிக்கை**
மே மாதத்தில் நடாலியா ஏப்ரல் மாதத்தில் விற்ற கிளிப்களின் **பாதி** (1/2) விற்றார்.
எனவே, மே மாதத்தில் விற்ற கிளிப்களின் எண்ணிக்கை = $48 \times \frac{1}{2}$
$$48 \div 2 = 24$$
எனவே, மே மாதத்தில் விற்ற கிளிப்களின் எண்ணிக்கை = **24**.

### **படி 3: இரண்டு மாதங்களில் விற்ற கிளிப்களின் மொத்த எண்ணிக்கை**
இரண்டு மாதங்களில் விற்ற கிளிப்களின் மொத்த எண்ணிக்கையைக் கணக்கிட, ஏப்ரல் மற்றும் மே மாதங்களில் விற்ற கிளிப்களின் எண்ணிக்கையைச் சேர்த்துக்கொள்

In [14]:
import torch
import json
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

MODEL_ID = "Qwen/Qwen3.5-2B"
DATASET_ID = "Vishal0407/GSM8K_TAMIL"

N_SAMPLES = 500
BATCH_SIZE = 4
MAX_NEW_TOKENS = 768
SEED = 42

OUTPUT_DIR = Path("/content/qwen35_gsm8k_tamil_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "predictions.jsonl"

eval_ds = ds.shuffle(seed=SEED).select(
    range(min(N_SAMPLES, len(ds)))
)

processor.tokenizer.padding_side = "left"

# Load previous predictions to support resuming.
done = {}
if PREDICTIONS_PATH.exists():
    with PREDICTIONS_PATH.open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            done[row["row_index"]] = row

pending = [i for i in range(len(eval_ds)) if i not in done]

print(f"Subset: {len(eval_ds)}")
print(f"Already completed: {len(done)}")
print(f"Remaining: {len(pending)}")

start_time = time.perf_counter()
new_examples = 0

with PREDICTIONS_PATH.open("a", encoding="utf-8") as f:
    for start in tqdm(range(0, len(pending), BATCH_SIZE)):
        indices = pending[start:start + BATCH_SIZE]
        samples = [eval_ds[i] for i in indices]

        messages = []
        for sample in samples:
            prompt = f"""கீழே உள்ள கணிதக் கேள்வியைத் தமிழில் படிப்படியாகத் தீர்க்கவும்.
இறுதி எண்ணியல் விடையை #### <number> என்ற வடிவில் கொடுக்கவும்.

கேள்வி:
{sample["question"]}
"""
            messages.append([
                {"role": "user", "content": prompt}
            ])

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            padding=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        input_width = inputs["input_ids"].shape[1]
        batch_start = time.perf_counter()

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

        elapsed = time.perf_counter() - batch_start

        generated_ids = output_ids[:, input_width:]
        responses = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        for idx, sample, response in zip(
            indices, samples, responses
        ):
            gold = extract_gold_answer(sample["answer"])
            predicted = extract_predicted_answer(response)

            row = {
                "row_index": idx,
                "question": sample["question"],
                "gold_answer": gold,
                "prediction": response,
                "predicted_answer": predicted,
                "correct": (
                    predicted == gold
                    if predicted is not None and gold is not None
                    else False
                ),
                "answer_extracted": predicted is not None,
                "batch_latency_seconds": elapsed,
                "generated_tokens": int(
                    generated_ids[
                        indices.index(idx)
                    ].shape[0]
                ),
            }

            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            done[idx] = row
            new_examples += 1

        f.flush()

elapsed_total = time.perf_counter() - start_time

print(f"New examples evaluated: {new_examples}")
print(f"Time this run: {elapsed_total / 60:.2f} minutes")
print(f"Predictions saved to: {PREDICTIONS_PATH}")

Subset: 500
Already completed: 0
Remaining: 500


  0%|          | 0/125 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

New examples evaluated: 500
Time this run: 76.14 minutes
Predictions saved to: /content/qwen35_gsm8k_tamil_benchmark/predictions.jsonl


In [15]:
import json
import pandas as pd
from pathlib import Path

PREDICTIONS_PATH = Path(
    "/content/qwen35_gsm8k_tamil_benchmark/predictions.jsonl"
)

with PREDICTIONS_PATH.open(encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)

# Use the saved predictions and gold answers.
df["gold_answer"] = df["gold_answer"].astype("string")
df["predicted_answer"] = df["predicted_answer"].astype("string")

df["answer_extracted"] = (
    df["predicted_answer"].notna()
    & df["predicted_answer"].str.strip().ne("")
    & df["predicted_answer"].str.lower().ne("none")
)

df["correct"] = (
    df["answer_extracted"]
    & df["gold_answer"].notna()
    & df["predicted_answer"].eq(df["gold_answer"])
)

n = len(df)
n_extracted = int(df["answer_extracted"].sum())
n_correct = int(df["correct"].sum())

metrics = {
    "Evaluated examples": n,
    "Correct answers": n_correct,
    "Numeric exact-match accuracy (%)": round(
        100 * n_correct / n, 2
    ) if n else 0.0,
    "Answer extraction coverage (%)": round(
        100 * n_extracted / n, 2
    ) if n else 0.0,
    "Accuracy among extracted answers (%)": round(
        100 * n_correct / n_extracted, 2
    ) if n_extracted else 0.0,
    "Unparseable predictions": n - n_extracted,
    "Incorrect extracted answers": n_extracted - n_correct,
}

display(pd.DataFrame(metrics.items(), columns=["Metric", "Value"]))

# Inspect examples where the model failed.
wrong = df[~df["correct"]]

display(
    wrong[
        ["question", "gold_answer", "predicted_answer", "prediction"]
    ].head(20)
)

# Save the evaluation report.
output_path = PREDICTIONS_PATH.parent / "evaluation_metrics.csv"
pd.DataFrame(metrics.items(), columns=["Metric", "Value"]).to_csv(
    output_path, index=False
)

print("Metrics saved to:", output_path)

,Metric,Value
0,Evaluated examples,500.0
1,Correct answers,152.0
2,Numeric exact-match accuracy (%),30.4
3,Answer extraction coverage (%),100.0
4,Accuracy among extracted answers (%),30.4
5,Unparseable predictions,0.0
6,Incorrect extracted answers,348.0


,question,gold_answer,predicted_answer,prediction
0,ஒவ்வொரு வாகனத்திற்கும் ஒரு வாகன நிறுத்துமிடத்த...,24,2,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."
1,ஆமி ஒரு வரலாற்று பரிசோதனையை எடுத்து வருகிறார்....,41,1,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, முதலில் ஆமி ..."
2,ஒரு கடை ஒரு குறிப்பிட்ட பிராண்ட் ஜீன்ஸ் $ 40 க...,112,216,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."
4,"ஜானின் பிக்கி வங்கியில் காலாண்டுகள், டைம்கள் ம...",22,2101,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."
5,ரோங் ஒவ்வொரு மாதமும் தனது பிக்கி வங்கியில் 20 ...,5760,3360,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."
7,கரேன் தனது பையுடனும் நீண்ட தூர உயர்வுக்காக பேக...,34,4,கீழே உள்ள கேள்வியைத் தமிழில் படிப்படியாகத் தீர...
8,பிரையன் புத்தகங்களைப் படிக்க விரும்புகிறார். ...,75,127,"கேள்வியைத் தீர்க்க, முதலில் பிரையன் படித்த அத்..."
9,பில் ஏப்ரல் முதல் நாளில் ஒரு புத்தகத்தைப் படிக...,144,8,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."
11,ட்ரே ஒரு புதிய பைக்கிற்கு 2 112 செலவாகும். அடு...,8,1,"கேள்வியைத் தீர்க்க, நாம் பின்வரும் படிகளைப் பி..."
14,நீங்கள் அழகிய வளையல்களை விற்கும் வணிகத்தைத் தொ...,50,125,"கொடுக்கப்பட்ட கேள்வியைத் தீர்க்க, படிப்படியாக ..."


Metrics saved to: /content/qwen35_gsm8k_tamil_benchmark/evaluation_metrics.csv


In [18]:
print(ta_ds[0]["answer"])
print(extract_gold_answer(ta_ds[0]["answer"]))

Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
18


In [19]:
gold = extract_gold_answer(sample["answer"])

In [20]:
metrics

{'Evaluated examples': 500,
 'Correct answers': 152,
 'Numeric exact-match accuracy (%)': 30.4,
 'Answer extraction coverage (%)': 100.0,
 'Accuracy among extracted answers (%)': 30.4,
 'Unparseable predictions': 0,
 'Incorrect extracted answers': 348}